# 3 Baseline Multi-Label Classifier

As a baseline, we trained a simple multi-label text classifier using TF-IDF features and silver labels generated from keyword matching. The model is a feed-forward neural network with sigmoid outputs and binary cross-entropy loss, optimized using Adam and early stopping based on validation micro-F1. This baseline allows us to assess whether the automatically generated silver labels contain meaningful signal and establishes a reference point for future improvements such as pseudo-labeling or hierarchy-aware training.

In [1]:
# Relevant imports
import os
import csv
import copy
import random
from tqdm import tqdm
from collections import defaultdict
from pathlib import Path
import numpy as np
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
import copy
from sklearn.metrics import f1_score
import torch
import torch.nn.functional as F

# Random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Setup initial filepaths
ROOT = Path("project_release")

TRAIN_CORPUS_PATH = ROOT / "Amazon_products" / "train" / "train_corpus.txt"
TEST_CORPUS_PATH = ROOT / "Amazon_products" / "test" / "test_corpus.txt"
SILVER_PATH = ROOT / "silver_labels"

SUBMISSION_PATH = ROOT / "submissions"
SUBMISSION_PATH.mkdir(exist_ok=True)

# Load Silver Labels
# Uncomment to use

# TF-IDF silver labels
with open(SILVER_PATH / "silver_labels_tfidf.pkl", "rb") as f:
    silver_labels = pickle.load(f)

# SVD silver labels
#with open(SILVER_PATH / "silver_labels_svd.pkl", "rb") as f:
#    silver_labels = pickle.load(f)

# BERT silver labels
#with open(SILVER_PATH / "silver_labels_bert.pkl", "rb") as f:
#    silver_labels = pickle.load(f)
    
# Hybrid silver labels
# with open(SILVER_PATH / "silver_labels_hybrid.pkl", "rb") as f:
#     silver_labels = pickle.load(f)


In [2]:
with open(TRAIN_CORPUS_PATH, "r", encoding="utf8") as f:
    train_texts = [line.strip() for line in f]

label_set = sorted({cls for _, labels in silver_labels for cls, _ in labels})
label_to_idx = {lbl: i for i, lbl in enumerate(label_set)}
num_labels = len(label_set)

y = np.zeros((len(train_texts), num_labels), dtype=np.float32)
for i, (_, labels) in enumerate(silver_labels):
    for cls, _ in labels:
        y[i, label_to_idx[cls]] = 1.0

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(train_texts).toarray().astype(np.float32)

class SilverDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return {
            "X": self.X[idx],
            "y": self.y[idx]
        }

dataset = SilverDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

class MLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = MLP(X.shape[1], num_labels).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [3]:
# ---------- Training loop ----------
EPOCHS = 10
patience = 5
best_val_f1 = -1
patience_counter = 0
best_model_state = None

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        X, y = batch["X"].to(device), batch["y"].to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"[Epoch {epoch}] Train Loss: {avg_loss:.4f}")

    # ---------- Validation ----------
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            X, y = batch["X"].to(device), batch["y"].to(device)
            logits = model(X)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()
            all_preds.append(preds.cpu())
            all_labels.append(y.cpu().int())

    all_preds = torch.cat(all_preds, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()
    val_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    print(f"[VAL] f1_micro: {val_f1:.4f}")

    # ---------- Early stopping ----------
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("[Early Stopping] No improvement.")
        break

# Save best model
torch.save(best_model_state, SILVER_PATH / "last_best.pth")
print("[DONE] Training complete. Best f1_micro:", best_val_f1)


[Epoch 1] Train Loss: 0.0453
[VAL] f1_micro: 0.0000
[Epoch 2] Train Loss: 0.0011
[VAL] f1_micro: 0.0000
[Epoch 3] Train Loss: 0.0010
[VAL] f1_micro: 0.0952
[Epoch 4] Train Loss: 0.0008
[VAL] f1_micro: 0.1818
[Epoch 5] Train Loss: 0.0006
[VAL] f1_micro: 0.2979
[Epoch 6] Train Loss: 0.0005
[VAL] f1_micro: 0.2418
[Epoch 7] Train Loss: 0.0004
[VAL] f1_micro: 0.2766
[Epoch 8] Train Loss: 0.0003
[VAL] f1_micro: 0.2340
[Epoch 9] Train Loss: 0.0002
[VAL] f1_micro: 0.2526
[Epoch 10] Train Loss: 0.0002
[VAL] f1_micro: 0.2340
[Early Stopping] No improvement.
[DONE] Training complete. Best f1_micro: 0.2978723404255319


In [4]:
model.load_state_dict(torch.load(SILVER_PATH / "last_best.pth", map_location=device))
model.eval()

with open(TEST_CORPUS_PATH, "r", encoding="utf8") as f:
    pid_text_pairs = []
    for line in f:
        pid, text = line.strip().split("\t", 1)
        pid_text_pairs.append((pid, text))

test_pids = [p for p, _ in pid_text_pairs]
test_texts = [t for _, t in pid_text_pairs]

X_test = vectorizer.transform(test_texts).toarray().astype(np.float32)
X_test = torch.from_numpy(X_test).to(device)

idx_to_label = {v: k for k, v in label_to_idx.items()}

all_preds = []
with torch.no_grad():
    for i in range(0, len(X_test), 256):
        logits = model(X_test[i:i+256])
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).cpu().numpy()
        all_preds.append(preds)

all_preds = np.vstack(all_preds)

submission_rows = []
for pid, pred_vec in zip(test_pids, all_preds):
    label_indices = np.where(pred_vec == 1)[0]
    if len(label_indices) == 0:
        label_indices = [int(np.argmax(pred_vec))]
    submission_rows.append(
        (pid, ",".join(str(idx) for idx in sorted(label_indices)))
    )

submission_file = SUBMISSION_PATH / "submission.csv"
with open(submission_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "labels"])
    for row in submission_rows:
        writer.writerow(row)

print(f"[DONE] Submission saved to {submission_file}")
print(f"Total samples: {len(submission_rows)}")


[DONE] Submission saved to project_release/submissions/submission.csv
Total samples: 19658
